# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [43]:
import numpy as np
import tensorflow as tf
import collections
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [44]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters
       生成一个由大小写字母组成的随机字符串 """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['BQKOSPCHBH', 'PNILWCIWPM'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 2, 17, 11, 15, 19, 16,  3,  8,  2,  8],
       [16, 14,  9, 12, 23,  3,  9, 23, 16, 13]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  8,  2,  8,  3, 16, 19, 15, 11, 17],
       [ 0, 13, 16, 23,  9,  3, 23, 12,  9, 14]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 8,  2,  8,  3, 16, 19, 15, 11, 17,  2],
       [13, 16, 23,  9,  3, 23, 12,  9, 14, 16]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [ ]:
class mySeq2SeqModel(tf.keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        todo
        
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        enc_ids: (b, enc_len)是编码器输入的token id，
        dec_ids: (b, dec_len)是解码器输入的token id
        '''
        # --- 编码器 ---
        enc_emb = self.embed_layer(enc_ids)  # (b, enc_len, v_sz)
        enc_out, enc_state = self.encoder(enc_emb)  # enc_out: (b, enc_len, hidden)
        
        # 解码器初始隐藏态取编码器最后一个时间步输出
        dec_state = enc_out[:, -1, :]  # (b, hidden)

        # 解码器输入嵌入
        dec_emb = self.embed_layer(dec_ids)  # (b, dec_len, emb)

        # --- 解码器 ---
        dec_out, _ = self.decoder(dec_emb, initial_state=dec_state)  # (b, dec_len, hidden)

        # --- 注意力机制 ---
        # 计算注意力权重
        proj_enc = self.dense_attn(enc_out)  # (b, enc_len, hidden)这个是对编码器输出进行线性变换后的结果，用于计算注意力分数
        attn_scores = tf.matmul(dec_out, proj_enc, transpose_b=True)  # (b, dec_len, enc_len)
        att_w = tf.nn.softmax(attn_scores, axis=-1)                 # (b, dec_len, enc_len)
        # 计算 context: (b, dec_len, hidden)
        context = tf.matmul(att_w, enc_out)   

        # 拼接并投影为 logits
        fusion = tf.concat([dec_out, context], axis=-1)    # (b, tgt_len, 2h)
        logits = self.dense(fusion)                         # (b, dec_len, hidden)
        

        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
        '''
        todo
        参考sequence_reversal-exercise, 自己构建单步解码逻辑'''

        x_emb = self.embed_layer(x)                       # (b, emb)
        dec_out, state = self.decoder_cell.call(x_emb, state)  # dec_h: (b, h)
        
        query = tf.expand_dims(dec_out, axis=1)              # (b, 1, h)
        proj_enc = self.dense_attn(enc_out)                # (b, src_len, h)
        attn_logits = tf.matmul(query, proj_enc, transpose_b=True)  # (b, 1, src_len)
        attn_weight = tf.nn.softmax(attn_logits, axis=-1)
        context = tf.matmul(attn_weight, enc_out)          # (b, 1, h)
        context = tf.squeeze(context, axis=1)              # (b, h)
        
        fusion = tf.concat([dec_out, context], axis=-1)      # (b, 2h)
        logits = self.dense(fusion)
        out = tf.argmax(logits, axis=-1, output_type=tf.int32)
        return out, state

# Loss函数以及训练逻辑

In [46]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)

        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [47]:
optimizer = tf.keras.optimizers.Adam(0.0005)
model = mySeq2SeqModel()
_, warmup_enc_x, warmup_dec_x, _ = get_batch(1, 20)
_ = model(warmup_enc_x, warmup_dec_x)
train(model, optimizer, seqlen=20)

step 0 : loss 3.3019722
step 500 : loss 1.6264784
step 1000 : loss 0.7140723
step 1500 : loss 0.24871373


<tf.Tensor: shape=(), dtype=float32, numpy=0.08632498979568481>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [48]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[False, True, True, True, True, True, True, False, True, True, False, True, False, True, True, True, True, False, True, True, False, False, True, True, True, True, True, True, False, True, False, False]
[('NOXDCAJLAUQGETDURWYD', 'DYWRUDTEGQUALJACDXON'), ('AXGIAYGFBCFMKKMVBWTT', 'TTWBVMKKMFCBFGYAIGXA'), ('MXIOMGCFODKKVLLHEGOK', 'MOGEHLLVKKDOFCGMOIXN'), ('BWXDNWXWZFUKRGXMUNSR', 'RSNUMXGRKUFZWXWNDXWB'), ('NUFPQVBHJYFXGSVPHDCB', 'BCDHPVSGXFYJHBVQPFUN'), ('PZZVRENHQPQNFLSFURLN', 'NLRUFSLFNQPQHNERVZZP'), ('PFKBKVLHBGCZYBNWBEQM', 'MQEBWNBYZCGBHLVKBTFP'), ('FHTVQIXIETYNFPMFHHOF', 'FOHHFMPFNYTEIXIQVTHW'), ('VIEUGDOUEIMTKHKYFEVR', 'RVEFYKHKTMIEUODGUEIV'), ('SNZPGDODVCYZYSNNBEDN', 'NDEBNNSYZYCVDODGPZRS'), ('KACBMUJPEVTEAHJVLNCZ', 'ZCNLVJHAETVEPJUMBCAK'), ('KVIUNFIAWZFYDHEOMUMJ', 'JMUMOEHDYFZWAIFNUIVK'), ('LOIJYXDUKVXRRUJQELOG', 'GOLEQJURRXVKUDXYJIOL'), ('ZUHCTMCOGWNWNMDDHGHZ', 'ZHGHDDMNWNWGOCMTCHUZ'), ('KUGTMBYJBQNPVEQUPIUU', 'UUIPUQEVPNQBJYBMTGWK'), ('AKKVZSLJCKLHKQPPNPTP', 'PTPNPPQKHLKCJLSZVKKA